In [ ]:
# # N-RDD2024 Dataset Analysis
# Comprehensive statistical analysis of the N-RDD2024 road damage dataset.
# Reads YOLO `.txt` label files from all country splits and produces plots
# covering class distribution, bounding box geometry, spatial density,
# annotation load, class co-occurrence, and COCO size categories.


In [ ]:
import os
import glob
import yaml
from collections import defaultdict
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# ── Kaggle input path ──────────────────────────────────────────────────────
DATASET_ROOT = Path("/kaggle/input/n-rdd2024/N-RDD2024Road damage and defects")
OUTPUT_DIR   = Path("/kaggle/working/plots")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────────
FIGURE_DPI = 120
plt.rcParams.update({"figure.dpi": FIGURE_DPI, "font.size": 10})


## 1. Discover available country/split combinations


In [ ]:
COUNTRIES = ["Czech Republic_txt", "USA_txt", "china-motorbike_txt", "india_txt"]

splits_found = {}
for country in COUNTRIES:
    country_path = DATASET_ROOT / country
    for split in ["train", "valid"]:
        labels_dir = country_path / split / "labels"
        if labels_dir.exists():
            splits_found.setdefault(country, []).append(split)
            print(f"  Found: {country}/{split}/labels  ({len(list(labels_dir.glob('*.txt')))} label files)")


## 2. Load class names from data.yaml


In [ ]:
CLASS_NAMES = None
for country in COUNTRIES:
    yaml_path = DATASET_ROOT / country / "data.yaml"
    if yaml_path.exists():
        with open(yaml_path) as f:
            cfg = yaml.safe_load(f)
        CLASS_NAMES = cfg.get("names", [])
        print(f"Classes from {yaml_path.name}: {CLASS_NAMES}")
        break

if CLASS_NAMES is None:
    # RDD2022 standard classes
    CLASS_NAMES = ["D00", "D10", "D20", "D40"]
    print(f"Using default classes: {CLASS_NAMES}")

NUM_CLASSES = len(CLASS_NAMES)


## 3. Parse all YOLO label files


In [ ]:
CLASS_COLORS = plt.cm.tab10(np.linspace(0, 0.9, NUM_CLASSES))

annotations = []   # list of dicts: class_id, cx, cy, w, h, country, split

for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for label_file in labels_dir.glob("*.txt"):
            with open(label_file) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    class_id = int(parts[0])
                    cx, cy, w, h = map(float, parts[1:5])
                    if w <= 0 or h <= 0:
                        continue
                    annotations.append({
                        "class_id": class_id,
                        "cx": cx, "cy": cy,
                        "w": w,   "h": h,
                        "area": w * h,
                        "aspect": w / h,
                        "compactness": (4 * np.pi * w * h) / (2 * (w + h)) ** 2,
                        "country": country.replace("_txt", ""),
                        "split": split,
                    })

print(f"Total annotations loaded: {len(annotations):,}")

# Convert to arrays per class
arrays = {i: defaultdict(list) for i in range(NUM_CLASSES)}
for ann in annotations:
    cid = ann["class_id"]
    if cid >= NUM_CLASSES:
        continue
    for key in ["cx", "cy", "w", "h", "area", "aspect", "compactness"]:
        arrays[cid][key].append(ann[key])

arrays = {cid: {k: np.array(v) for k, v in d.items()} for cid, d in arrays.items()}


## 4. Summary statistics


In [ ]:
print(f"\n{'Class':<20} {'Count':>8}  {'Mean W':>8}  {'Mean H':>8}  {'Mean Area':>10}")
print("-" * 60)
for cid, cname in enumerate(CLASS_NAMES):
    d = arrays[cid]
    n = len(d.get("w", []))
    if n == 0:
        print(f"  {cname:<18} {'0':>8}")
        continue
    print(f"  {cname:<18} {n:>8,}  {np.mean(d['w']):>8.4f}  {np.mean(d['h']):>8.4f}  {np.mean(d['area']):>10.5f}")


## Plot 01 — Class Instance Counts


In [ ]:
counts = [len(arrays[i].get("w", [])) for i in range(NUM_CLASSES)]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(CLASS_NAMES, counts, color=CLASS_COLORS, edgecolor="white")
ax.bar_label(bars, labels=[f"{c:,}" for c in counts], padding=6, fontsize=10, fontweight="bold")
ax.set_xlabel("Number of annotated instances")
ax.set_title("N-RDD2024 — Class Instance Counts", fontsize=13, fontweight="bold")
ax.invert_yaxis()
ax.set_xlim(0, max(counts) * 1.18)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_class_instance_counts.png", bbox_inches="tight")
plt.show()


## Plot 02 — Class Share Pie


In [ ]:
nonzero_idx    = [i for i, c in enumerate(counts) if c > 0]
nonzero_names  = [CLASS_NAMES[i] for i in nonzero_idx]
nonzero_counts = [counts[i] for i in nonzero_idx]
nonzero_colors = [CLASS_COLORS[i] for i in nonzero_idx]

fig, ax = plt.subplots(figsize=(7, 5))
wedges, texts, autotexts = ax.pie(
    nonzero_counts, labels=nonzero_names, colors=nonzero_colors,
    autopct="%1.1f%%", startangle=140, pctdistance=0.82,
    wedgeprops={"edgecolor": "white", "linewidth": 1.2},
)
for t in autotexts:
    t.set_fontsize(9); t.set_fontweight("bold")
ax.set_title(f"Class Distribution  (total: {sum(nonzero_counts):,} instances)",
             fontsize=13, fontweight="bold", pad=15)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_class_share_pie.png", bbox_inches="tight")
plt.show()


## Plot 03 — Country Contribution per Class


In [ ]:
countries = sorted({a["country"] for a in annotations})
x = np.arange(NUM_CLASSES)
width = 0.8 / max(len(countries), 1)
cmap = plt.cm.Set2(np.linspace(0, 1, len(countries)))

fig, ax = plt.subplots(figsize=(11, 5))
for i, country in enumerate(countries):
    src_counts = []
    for cid in range(NUM_CLASSES):
        n = sum(1 for a in annotations if a["class_id"] == cid and a["country"] == country)
        src_counts.append(n)
    offset = (i - len(countries) / 2 + 0.5) * width
    ax.bar(x + offset, src_counts, width * 0.9, label=country,
           color=cmap[i], edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=15, ha="right")
ax.set_ylabel("Instance count")
ax.set_title("Country Contribution per Class", fontsize=13, fontweight="bold")
ax.legend(title="Country", fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_country_contribution_per_class.png", bbox_inches="tight")
plt.show()


## Plot 04 — Annotations per Image


In [ ]:
ann_per_img = defaultdict(int)
for ann in annotations:
    ann_per_img[ann.get("country","") + "_" + ann.get("split","")] 

# Re-count properly from label files
img_ann_count = defaultdict(int)
for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for lf in labels_dir.glob("*.txt"):
            key = lf.stem
            with open(lf) as f:
                lines = [l for l in f if l.strip()]
            img_ann_count[key] = len(lines)

counts_per_img = np.array(list(img_ann_count.values()))
counts_nonzero = counts_per_img[counts_per_img > 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.hist(counts_per_img, bins=40, color="#3498DB", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(counts_per_img), color="#E74C3C", lw=2, linestyle="--",
           label=f"Mean: {np.mean(counts_per_img):.2f}")
ax.axvline(np.median(counts_per_img), color="#F39C12", lw=2, linestyle=":",
           label=f"Median: {np.median(counts_per_img):.2f}")
ax.set_xlabel("Annotations per image"); ax.set_ylabel("Number of images")
ax.set_title("Annotations per Image (all)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

ax2 = axes[1]
ax2.hist(counts_nonzero, bins=40, color="#2ECC71", edgecolor="white", alpha=0.85)
ax2.set_yscale("log")
ax2.set_xlabel("Annotations per image (annotated only)"); ax2.set_ylabel("Count (log)")
ax2.set_title("Annotations per Image (log y)", fontsize=12, fontweight="bold")

fig.suptitle("Annotation Load Distribution", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_annotations_per_image.png", bbox_inches="tight")
plt.show()


## Plot 05 — BBox Width Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cid, cname in enumerate(CLASS_NAMES):
    w = arrays[cid].get("w", np.array([]))
    if len(w) == 0: continue
    ax.hist(w, bins=80, alpha=0.55, label=cname, color=CLASS_COLORS[cid], edgecolor="none")
    ax.axvline(np.mean(w), color=CLASS_COLORS[cid], linestyle="--", linewidth=1.2, alpha=0.9)
ax.set_xlabel("Bounding box width (normalised)"); ax.set_ylabel("Count")
ax.set_title("Bounding Box Width Distribution per Class", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, title="Class")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_bbox_width_distribution.png", bbox_inches="tight")
plt.show()


## Plot 06 — BBox Height Distribution


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for cid, cname in enumerate(CLASS_NAMES):
    h = arrays[cid].get("h", np.array([]))
    if len(h) == 0: continue
    ax.hist(h, bins=80, alpha=0.55, label=cname, color=CLASS_COLORS[cid], edgecolor="none")
    ax.axvline(np.mean(h), color=CLASS_COLORS[cid], linestyle="--", linewidth=1.2, alpha=0.9)
ax.set_xlabel("Bounding box height (normalised)"); ax.set_ylabel("Count")
ax.set_title("Bounding Box Height Distribution per Class", fontsize=13, fontweight="bold")
ax.legend(fontsize=9, title="Class")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_bbox_height_distribution.png", bbox_inches="tight")
plt.show()


## Plot 07 — BBox Area Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for cid, cname in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    if len(areas) == 0: continue
    ax.hist(np.log10(areas + 1e-9), bins=60, alpha=0.5, label=cname,
            color=CLASS_COLORS[cid], edgecolor="none")
ax.set_xlabel("log₁₀(bbox area)"); ax.set_ylabel("Count")
ax.set_title("BBox Area (log scale)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

ax2 = axes[1]
data_box, labels_box = [], []
for cid, cname in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    if len(areas) > 0:
        data_box.append(np.log10(areas + 1e-9)); labels_box.append(cname)
if data_box:
    bp = ax2.boxplot(data_box, patch_artist=True,
                     medianprops={"color": "black", "linewidth": 2})
    for patch, cid in zip(bp["boxes"], [CLASS_NAMES.index(l) for l in labels_box]):
        patch.set_facecolor(CLASS_COLORS[cid]); patch.set_alpha(0.7)
    ax2.set_xticklabels(labels_box, rotation=20, ha="right", fontsize=9)
    ax2.set_ylabel("log₁₀(area)")
    ax2.set_title("BBox Area Box Plot per Class", fontsize=12, fontweight="bold")

fig.suptitle("Bounding Box Area Analysis", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_bbox_area_distribution.png", bbox_inches="tight")
plt.show()


## Plot 08 — Aspect Ratio Violin


In [ ]:
valid = [(cname, arrays[cid]["aspect"])
         for cid, cname in enumerate(CLASS_NAMES)
         if len(arrays[cid].get("aspect", [])) > 0]

if valid:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    ax = axes[0]
    names_v = [v[0] for v in valid]
    data_v  = [v[1] for v in valid]
    cids_v  = [CLASS_NAMES.index(n) for n in names_v]

    parts = ax.violinplot(data_v, showmedians=True, showextrema=False)
    for body, cid in zip(parts["bodies"], cids_v):
        body.set_facecolor(CLASS_COLORS[cid]); body.set_alpha(0.7)
    parts["cmedians"].set_color("black"); parts["cmedians"].set_linewidth(2)
    ax.set_xticks(range(1, len(names_v)+1))
    ax.set_xticklabels(names_v, rotation=20, ha="right", fontsize=9)
    ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, alpha=0.6, label="square (w=h)")
    ax.set_ylabel("width / height"); ax.set_title("Aspect Ratio per Class", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylim(0, min(np.percentile(np.concatenate(data_v), 99) * 1.3, 15))

    ax2 = axes[1]
    means = [np.mean(d) for d in data_v]
    stds  = [np.std(d)  for d in data_v]
    x = np.arange(len(names_v))
    ax2.bar(x, means, color=[CLASS_COLORS[i] for i in cids_v], alpha=0.8, edgecolor="white")
    ax2.errorbar(x, means, yerr=stds, fmt="none", color="black", capsize=5, linewidth=1.5)
    ax2.axhline(1.0, color="gray", linestyle="--", linewidth=1, alpha=0.6)
    ax2.set_xticks(x); ax2.set_xticklabels(names_v, rotation=20, ha="right", fontsize=9)
    ax2.set_ylabel("Mean aspect ratio ± std")
    ax2.set_title("Mean Aspect Ratio per Class", fontsize=12, fontweight="bold")

    fig.suptitle("Aspect Ratio Analysis", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "08_aspect_ratio_violin.png", bbox_inches="tight")
    plt.show()


## Plot 09 — Spatial Density Heatmap


In [ ]:
GRID = 32
centres = {cid: ([], []) for cid in range(NUM_CLASSES)}
for ann in annotations:
    cid = ann["class_id"]
    if cid < NUM_CLASSES:
        centres[cid][0].append(ann["cx"])
        centres[cid][1].append(ann["cy"])

active = [cid for cid in range(NUM_CLASSES) if len(centres[cid][0]) > 0]
cols = min(len(active), 3)
rows = (len(active) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4.5*rows))
axes = np.array(axes).flatten() if len(active) > 1 else [axes]

for ax, cid in zip(axes, active):
    cx = np.array(centres[cid][0])
    cy = np.array(centres[cid][1])
    heatmap, _, _ = np.histogram2d(cx, cy, bins=GRID, range=[[0,1],[0,1]])
    im = ax.imshow(heatmap.T, origin="upper", aspect="equal",
                   cmap="YlOrRd", interpolation="bilinear", extent=[0,1,1,0])
    plt.colorbar(im, ax=ax, shrink=0.8, label="Count")
    ax.set_title(CLASS_NAMES[cid], fontsize=10, fontweight="bold")
    ax.set_xlabel("Normalised x"); ax.set_ylabel("Normalised y")
    ax.tick_params(labelsize=7)

for ax in axes[len(active):]:
    ax.axis("off")

fig.suptitle("Spatial Density of Annotation Centres", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_spatial_density_heatmap.png", bbox_inches="tight")
plt.show()


## Plot 10 — COCO Size Categories


In [ ]:
# YOLO areas are normalised; thresholds relative to a 640x640 image
SMALL_MAX  = (32/640)**2
MEDIUM_MAX = (96/640)**2

active_names = [CLASS_NAMES[cid] for cid in range(NUM_CLASSES) if len(arrays[cid].get("area",[])) > 0]
active_ids   = [cid for cid in range(NUM_CLASSES) if len(arrays[cid].get("area",[])) > 0]

small_c, medium_c, large_c = [], [], []
for cid in active_ids:
    a = arrays[cid]["area"]
    small_c.append(int(np.sum(a < SMALL_MAX)))
    medium_c.append(int(np.sum((a >= SMALL_MAX) & (a < MEDIUM_MAX))))
    large_c.append(int(np.sum(a >= MEDIUM_MAX)))

x = np.arange(len(active_names)); w = 0.25
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
for i, (cnts, lbl, clr) in enumerate(zip(
        [small_c, medium_c, large_c],
        ["Small (<32px @ 640)", "Medium (32–96px)", "Large (>96px)"],
        ["#3498DB", "#F39C12", "#E74C3C"])):
    ax.bar(x + (i-1)*w, cnts, w, label=lbl, color=clr, alpha=0.8, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(active_names, rotation=15, ha="right", fontsize=9)
ax.set_ylabel("Instance count"); ax.set_title("COCO Size Categories (grouped)", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))

ax2 = axes[1]
totals = [s+m+l for s,m,l in zip(small_c, medium_c, large_c)]
s_p = [s/t*100 if t else 0 for s,t in zip(small_c, totals)]
m_p = [m/t*100 if t else 0 for m,t in zip(medium_c, totals)]
l_p = [l/t*100 if t else 0 for l,t in zip(large_c, totals)]
ax2.bar(x, s_p, color="#3498DB", alpha=0.8, label="Small", edgecolor="white")
ax2.bar(x, m_p, bottom=s_p, color="#F39C12", alpha=0.8, label="Medium", edgecolor="white")
ax2.bar(x, l_p, bottom=[s+m for s,m in zip(s_p,m_p)], color="#E74C3C", alpha=0.8, label="Large", edgecolor="white")
ax2.set_xticks(x); ax2.set_xticklabels(active_names, rotation=15, ha="right", fontsize=9)
ax2.set_ylabel("Percentage (%)"); ax2.set_ylim(0,100)
ax2.set_title("COCO Size Categories (normalised)", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)

fig.suptitle("COCO Size Category Analysis", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_coco_size_categories.png", bbox_inches="tight")
plt.show()


## Plot 11 — Class Co-occurrence Heatmap


In [ ]:
img_classes = defaultdict(set)
for ann in annotations:
    key = ann.get("country","") + "_" + ann.get("split","") + "_" + str(id(ann))
for country, splits in splits_found.items():
    for split in splits:
        labels_dir = DATASET_ROOT / country / split / "labels"
        for lf in labels_dir.glob("*.txt"):
            img_key = country + "_" + split + "_" + lf.stem
            with open(lf) as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cid = int(parts[0])
                        if cid < NUM_CLASSES:
                            img_classes[img_key].add(cid)

matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for classes in img_classes.values():
    clist = list(classes)
    for i in clist:
        for j in clist:
            matrix[i, j] += 1

diag = matrix.diagonal().astype(float); diag[diag == 0] = 1
norm_matrix = matrix / diag[:, None]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(matrix, annot=True, fmt=",d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[0], cbar_kws={"label": "Count"})
axes[0].set_title("Class Co-occurrence (raw)", fontsize=12, fontweight="bold")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha="right", fontsize=8)

sns.heatmap(norm_matrix, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=axes[1], vmin=0, vmax=1,
            cbar_kws={"label": "P(col|row)"})
axes[1].set_title("Class Co-occurrence (normalised)", fontsize=12, fontweight="bold")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right", fontsize=8)

fig.suptitle("Class Co-occurrence per Image", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_class_cooccurrence_heatmap.png", bbox_inches="tight")
plt.show()


## Plot 12 — Compactness vs Area Scatter


In [ ]:
MAX_POINTS = 2000
fig, ax = plt.subplots(figsize=(10, 5))
for cid, cname in enumerate(CLASS_NAMES):
    areas = arrays[cid].get("area", np.array([]))
    comp  = arrays[cid].get("compactness", np.array([]))
    if len(areas) == 0: continue
    n = len(areas)
    if n > MAX_POINTS:
        idx = np.random.choice(n, MAX_POINTS, replace=False)
        areas = areas[idx]; comp = comp[idx]
    ax.scatter(np.log10(areas + 1e-9), comp, c=[CLASS_COLORS[cid]],
               alpha=0.35, s=10, label=f"{cname} (n={n:,})", edgecolors="none")

ax.set_xlabel("log₁₀(bbox area)"); ax.set_ylabel("Compactness (4π·A/P²)")
ax.set_title("BBox Compactness vs Area\n(1=circle, ~0.05=elongated crack)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8, markerscale=2, title="Class")
ax.axhline(0.785, color="gray", linestyle=":", linewidth=1, alpha=0.5)
ax.text(ax.get_xlim()[0]+0.05, 0.80, "square ≈ 0.785", fontsize=7, color="gray")
ax.set_ylim(0, 1.05)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_compactness_vs_area_scatter.png", bbox_inches="tight")
plt.show()


## Summary


In [ ]:
plots = sorted(OUTPUT_DIR.glob("*.png"))
print(f"\n✅ {len(plots)} plots saved to {OUTPUT_DIR}")
for p in plots:
    print(f"  {p.name}")
